# End-to-End Ontology Alignment: OBO + OWL + SSSOM

This tutorial aligns two disease ontologies by merging three complementary data sources:

| File | Format | What it provides |
|---|---|---|
| `mondo_subset.obo` | OBO | MONDO disease hierarchy + xref mappings to ORDO |
| `ordo_subset.ofn` | OWL Functional Syntax | ORDO rare disease hierarchy + **disjointness** constraints |
| `mondo_ordo_mappings.sssom.tsv` | SSSOM | Cross-ontology equivalence candidates with confidence scores |

In [1]:
from pathlib import Path
from IPython.display import Markdown

DIR = Path("docs/tutorial/ontology-alignment-files")

def show(path, lang="yaml"):
    """Display a file as a fenced code block."""
    text = (DIR / path).read_text()
    return Markdown(f"**`{path}`**\n\n```{lang}\n{text}```")

## Input Files

### MONDO hierarchy (OBO)

Three diseases under a common root, with xrefs to ORDO for the first two:

In [2]:
show("mondo_subset.obo")

**`mondo_subset.obo`**

```yaml
format-version: 1.4
ontology: mondo-subset

[Term]
id: MONDO:0000001
name: disease

[Term]
id: MONDO:0001234
name: alpha disease
is_a: MONDO:0000001 ! disease
xref: ORDO:100

[Term]
id: MONDO:0005678
name: beta disease
is_a: MONDO:0000001 ! disease
xref: ORDO:200

[Term]
id: MONDO:0009999
name: gamma disease
is_a: MONDO:0000001 ! disease
```

### ORDO hierarchy (OWL)

Three rare diseases under a grouping class. Crucially, they are declared **pairwise disjoint**:

In [3]:
show("ordo_subset.ofn", lang="turtle")

**`ordo_subset.ofn`**

```turtle
Prefix(ORDO:=<http://www.orpha.net/ORDO/Orphanet_>)
Prefix(owl:=<http://www.w3.org/2002/07/owl#>)
Prefix(rdfs:=<http://www.w3.org/2000/01/rdf-schema#>)

Ontology(<http://www.orpha.net/ORDO/ordo-subset>

Declaration(Class(ORDO:100))
Declaration(Class(ORDO:200))
Declaration(Class(ORDO:300))
Declaration(Class(ORDO:999))

SubClassOf(ORDO:100 ORDO:999)
SubClassOf(ORDO:200 ORDO:999)
SubClassOf(ORDO:300 ORDO:999)

DisjointClasses(ORDO:100 ORDO:200 ORDO:300)

AnnotationAssertion(rdfs:label ORDO:100 "Alpha rare disease")
AnnotationAssertion(rdfs:label ORDO:200 "Beta rare disease")
AnnotationAssertion(rdfs:label ORDO:300 "Delta rare disease")
AnnotationAssertion(rdfs:label ORDO:999 "Rare disease grouping")
)
```

### Cross-ontology mappings (SSSOM)

Four candidate equivalences. Note the **conflicting** last row — MONDO:0009999 has a weak match (0.3) to ORDO:100, which is already strongly matched to MONDO:0001234:

In [4]:
show("mondo_ordo_mappings.sssom.tsv", lang="tsv")

**`mondo_ordo_mappings.sssom.tsv`**

```tsv
#curie_map:
#  MONDO: http://purl.obolibrary.org/obo/MONDO_
#  ORDO: http://www.orpha.net/ORDO/Orphanet_
#  skos: http://www.w3.org/2004/02/skos/core#
#  semapv: https://w3id.org/semapv/vocab/
#mapping_set_id: https://example.org/mondo-ordo-mappings
#mapping_set_description: MONDO to ORDO candidate mappings
subject_id	subject_label	predicate_id	object_id	object_label	mapping_justification	confidence
MONDO:0001234	alpha disease	skos:exactMatch	ORDO:100	Alpha rare disease	semapv:LexicalMatching	0.9
MONDO:0005678	beta disease	skos:exactMatch	ORDO:200	Beta rare disease	semapv:LexicalMatching	0.85
MONDO:0009999	gamma disease	skos:exactMatch	ORDO:300	Delta rare disease	semapv:LexicalMatching	0.6
MONDO:0009999	gamma disease	skos:exactMatch	ORDO:100	Alpha rare disease	semapv:LexicalMatching	0.3
```

## Merge and Solve

Pass all three files directly to `merge` — formats are auto-detected from extensions.
Then solve the merged KB:

In [5]:
%%bash
uv run python -m boomer.cli merge \
  docs/tutorial/ontology-alignment-files/mondo_subset.obo \
  docs/tutorial/ontology-alignment-files/ordo_subset.ofn \
  docs/tutorial/ontology-alignment-files/mondo_ordo_mappings.sssom.tsv \
  -o docs/tutorial/ontology-alignment-files/merged.yaml \
  -n "MONDO-ORDO Alignment"

Merged 3 files into docs/tutorial/ontology-alignment-files/merged.yaml (yaml)
Final KB contains 25 f

acts and 6 probabilistic facts


In [6]:
%%bash
uv run python -m boomer.cli solve \
  docs/tutorial/ontology-alignment-files/merged.yaml \
  --timeout 60 \
  -O markdown

Loaded KB with 25 facts and 6 probabilistic facts


Starting search...


Solving KB: MONDO-ORDO Alignment with 6 pfacts; threshold=200
Search completed in 0.1011 seconds
Kno

wledge Base: MONDO-ORDO Alignment
KB num pfacts: 6

Search Statistics:


  Confidence: 0.5000
  Prior Probability: 1.5744e-01
  Posterior Probability: 0.1111
  Combinations 

...

85 posterior: 0.9571896919792617
 * True MONDO:0001234 (alpha disease) ≡ ORDO:100 (Alpha rare dise

ase) :: prior: 0.7 posterior: 0.968219477482972
 * True MONDO:0005678 (beta disease) ≡ ORDO:200 (B

eta rare disease) :: prior: 0.7 posterior: 0.9571896919792617
 * True MONDO:0009999 (gamma disease) 

≡ ORDO:300 (Delta rare disease) :: prior: 0.6 posterior: 0.5972095150960656
 * False MONDO:0009999

 (gamma disease) ≡ ORDO:100 (Alpha rare disease) :: prior: 0.3 posterior: 0.004650808173223541



## Interpreting the Results

| Mapping | Prior | Posterior | Verdict | Why |
|---|---|---|---|---|
| MONDO:0001234 ≡ ORDO:100 | 0.90 | ~0.97 | **Accepted** | Reinforced by both xref and SSSOM |
| MONDO:0005678 ≡ ORDO:200 | 0.85 | ~0.96 | **Accepted** | Reinforced by both xref and SSSOM |
| MONDO:0009999 ≡ ORDO:300 | 0.60 | ~0.60 | **Accepted** | Moderate match, no competition |
| MONDO:0009999 ≡ ORDO:100 | 0.30 | ~0.005 | **Rejected** | Crushed by disjointness constraint |

The key result: the false mapping MONDO:0009999≡ORDO:100 drops from 0.30 to **0.005**.
This happens because ORDO:100 is already claimed by MONDO:0001234 (high confidence),
and the OWL `DisjointClasses` axiom makes it inconsistent for two MONDO terms to map to the same ORDO class.

## Export as TSV

In [7]:
%%bash
uv run python -m boomer.cli solve \
  docs/tutorial/ontology-alignment-files/merged.yaml \
  --timeout 60 -O tsv -o /tmp/solution.tsv --quiet

echo "=== Accepted ==="
grep "^EquivalentTo.*True" /tmp/solution.tsv | cut -f1-5
echo
echo "=== Rejected ==="
grep "^EquivalentTo.*False" /tmp/solution.tsv | cut -f1-5

Solving KB: MONDO-ORDO Alignment with 6 pfacts; threshold=200


=== Accepted ===


EquivalentTo	MONDO:0001234	ORDO:100	alpha disease	Alpha rare disease
EquivalentTo	MONDO:0005678	ORDO

:200	beta disease	Beta rare disease
EquivalentTo	MONDO:0001234	ORDO:100	alpha disease	Alpha rare dis

ease
EquivalentTo	MONDO:0005678	ORDO:200	beta disease	Beta rare disease
EquivalentTo	MONDO:0009999	O

RDO:300	gamma disease	Delta rare disease



=== Rejected ===


EquivalentTo	MONDO:0009999	ORDO:100	gamma disease	Alpha rare disease


## Python API Equivalent

In [8]:
from boomer.ontology_converter import obo_to_kb, owl_to_kb
from boomer.sssom_converter import sssom_to_kb
from boomer.search import solve, SearchConfig

mondo_kb = obo_to_kb(DIR / "mondo_subset.obo")
ordo_kb = owl_to_kb(DIR / "ordo_subset.ofn")
mapping_kb = sssom_to_kb(DIR / "mondo_ordo_mappings.sssom.tsv")

merged = mondo_kb.extend(
    facts=ordo_kb.facts + mapping_kb.facts,
    pfacts=ordo_kb.pfacts + mapping_kb.pfacts,
    labels={**ordo_kb.labels, **mapping_kb.labels},
)
merged.normalize()

solution = solve(merged, config=SearchConfig(timeout_seconds=60))

for sp in solution.solved_pfacts:
    f = sp.pfact.fact
    if f.fact_type == "EquivalentTo":
        verdict = "ACCEPTED" if sp.truth_value else "REJECTED"
        print(f"{verdict}: {f.sub} \u2261 {f.equivalent}  "
              f"(prior={sp.pfact.prob:.2f} \u2192 posterior={sp.posterior_prob:.3f})")

Solving KB: mondo-subset with 6 pfacts; threshold=200


ACCEPTED: MONDO:0001234 ≡ ORDO:100  (prior=0.90 → posterior=0.968)
ACCEPTED: MONDO:0005678 ≡ ORDO:200  (prior=0.85 → posterior=0.957)
ACCEPTED: MONDO:0001234 ≡ ORDO:100  (prior=0.70 → posterior=0.968)
ACCEPTED: MONDO:0005678 ≡ ORDO:200  (prior=0.70 → posterior=0.957)
ACCEPTED: MONDO:0009999 ≡ ORDO:300  (prior=0.60 → posterior=0.597)
REJECTED: MONDO:0009999 ≡ ORDO:100  (prior=0.30 → posterior=0.005)


## Summary

Three commands cover the entire workflow:

```bash
# Merge all sources (formats auto-detected)
pyboomer merge ontology.obo hierarchy.ofn mappings.sssom.tsv -o merged.yaml

# Solve
pyboomer solve merged.yaml -O markdown

# Export
pyboomer solve merged.yaml -O tsv -o solution.tsv --quiet
```

Structural constraints from ontologies (disjointness, hierarchy) interact with
probabilistic evidence from mappings to produce better alignments than either source alone.